# ReefScan — DINOv2-B coral-health classifier (one-shot)

Trains the 2-class (healthy / bleached) head on the NOAA-PIFSC bleaching dataset, then
**pushes weights + a conformal calibration to the Hub in the same run** so a dropped
session can't lose your work. Runs on **Colab** or **Kaggle** (auto-detected).

### How to use
1. Runtime → Change runtime type → **GPU**.
2. In the **Config** cell set `WANDB_KEY` and `HF_TOKEN` (and keep `PUSH_TO_HUB = True`).
3. Runtime → **Run all**.

Fast data path: pulls the dataset's **4 parquet shards** (~0.8 GB, a couple minutes) instead
of 10,419 individual PNGs (which took hours). Labels are recovered by joining the parquet
filenames to the repo's folder structure.

Linear probe ≈ 5–10 min on a Colab GPU. For the optional full fine-tune, set
`STAGE = "finetune"` and Run all again.


## 1. Install deps

In [ ]:
!pip -q install "transformers>=4.44" "huggingface_hub>=0.24" safetensors \
    "scikit-learn>=1.5" "wandb>=0.17" pyarrow


## 2. Config — EDIT THESE

In [ ]:
import os

STAGE        = "linear_probe"   # "linear_probe" first; rerun with "finetune" later
EPOCHS       = 10               # linear_probe ~10; finetune ~8
BATCH_SIZE   = 64
BACKBONE     = "facebook/dinov2-base"
UNFREEZE_LAST_N = 2             # finetune: trailing transformer blocks to unfreeze

WANDB_PROJECT = "reefscan"
WANDB_KEY     = ""              # <- paste your W&B key
HF_REPO       = "HrishiKabra/reefscan-dinov2-coral"
HF_TOKEN      = ""              # <- paste your HF write token
PUSH_TO_HUB   = True            # pushes weights + conformal.json at end of the run

RESUME_DIR    = None            # set to a prior checkpoints dir to resume across sessions

# checkpoint dir (auto: Kaggle / Colab+Drive / Colab / local)
if os.path.isdir("/kaggle/working"):
    WORK_DIR = "/kaggle/working"
elif os.path.isdir("/content"):
    WORK_DIR = "/content/drive/MyDrive/reefscan" if os.path.isdir("/content/drive/MyDrive") else "/content/reefscan"
else:
    WORK_DIR = "./reefscan_out"
CKPT_DIR = os.path.join(WORK_DIR, "checkpoints", STAGE)
os.makedirs(CKPT_DIR, exist_ok=True)
if HF_TOKEN: os.environ["HF_TOKEN"] = HF_TOKEN
print("work dir:", WORK_DIR, "| stage:", STAGE)


## 3. Fast data load (parquet shards + label join)

Downloads the 4 auto-converted parquet shards and recovers each image's `CORAL`/`CORAL_BL`
label by matching its filename against the repo's `split/LABEL/file.PNG` listing (filenames
are globally unique — verified). Images decode from the parquet bytes; no per-file download.

In [ ]:
import io
from collections import defaultdict
from pathlib import Path
import numpy as np, torch, torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import AutoModel
from huggingface_hub import HfApi, hf_hub_download
import pyarrow.parquet as pq

REPO_DS = "NMFS-OSI/NOAA-PIFSC-ESD-CORAL-Bleaching-Dataset"
CLASSES = ("healthy", "bleached")
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
LABEL_MAPPING = {"CORAL": "healthy", "CORAL_BL": "bleached"}

IMAGENET_MEAN, IMAGENET_STD, INPUT = (0.485,0.456,0.406), (0.229,0.224,0.225), 224
def build_tf(train):
    ops = [transforms.Resize((INPUT,INPUT))]
    if train: ops += [transforms.RandomHorizontalFlip(), transforms.ColorJitter(0.2,0.2)]
    ops += [transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)]
    return transforms.Compose(ops)

_api = HfApi()
def _basename_to_label():
    m = {}
    for f in _api.list_repo_files(REPO_DS, repo_type="dataset"):
        p = f.split("/")
        if len(p) >= 3 and f.lower().endswith(".png"): m[p[-1]] = p[1]
    return m
def _shards():
    out = defaultdict(list)
    for f in _api.list_repo_files(REPO_DS, repo_type="dataset", revision="refs/convert/parquet"):
        if f.endswith(".parquet"):
            sp = f.split("/")[-2]
            k = {"train":"train","validation":"val","test":"test"}.get(sp)
            if k: out[k].append(f)
    return out

_B2L = _basename_to_label()
_SHARDS = _shards()
_CACHE = {}
def _load_items(split):
    items = []
    for pf in _SHARDS[split]:
        path = hf_hub_download(REPO_DS, pf, repo_type="dataset", revision="refs/convert/parquet")
        for r in pq.read_table(path, columns=["image"]).column("image").to_pylist():
            m = LABEL_MAPPING.get(_B2L.get(r["path"]))
            if m is not None: items.append((r["bytes"], CLASS_TO_IDX[m]))
    return items

class CoralPatchDataset(Dataset):
    def __init__(self, split, train=None):
        train = (split == "train") if train is None else train
        self.tf = build_tf(train)
        if split not in _CACHE: _CACHE[split] = _load_items(split)
        self.items = _CACHE[split]
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        b, y = self.items[i]
        return self.tf(Image.open(io.BytesIO(b)).convert("RGB")), y

# prefetch + report
for s in ("train","val","test"):
    ds = CoralPatchDataset(s)
    import collections
    c = collections.Counter(y for _, y in ds.items)
    print(f"{s:5s}: {len(ds):5d}  healthy={c[0]}  bleached={c[1]}")


## 4. Model + freeze logic

In [ ]:
class DINOv2Classifier(nn.Module):
    def __init__(self, num_classes, backbone=BACKBONE):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(backbone)
        self.head = nn.Linear(self.backbone.config.hidden_size, num_classes)
    def forward(self, x):
        o = self.backbone(pixel_values=x)
        cls = getattr(o, "pooler_output", None)
        if cls is None: cls = o.last_hidden_state[:,0]
        return self.head(cls)

def set_trainable(model, stage):
    for p in model.backbone.parameters(): p.requires_grad = False
    for p in model.head.parameters(): p.requires_grad = True
    if stage == "linear_probe":
        return [{"params": model.head.parameters(), "lr": 1e-3}]
    for blk in model.backbone.encoder.layer[-UNFREEZE_LAST_N:]:
        for p in blk.parameters(): p.requires_grad = True
    if hasattr(model.backbone, "layernorm"):
        for p in model.backbone.layernorm.parameters(): p.requires_grad = True
    bb = [p for p in model.backbone.parameters() if p.requires_grad]
    return [{"params": model.head.parameters(), "lr": 1e-4}, {"params": bb, "lr": 1e-5}]


## 5. Checkpoint / resume (every epoch)

In [ ]:
import json, glob
def save_ckpt(model, opt, sched, scaler, epoch, best):
    blob = {"epoch":epoch,"best_f1":best,"model":model.state_dict(),"optimizer":opt.state_dict(),
            "scheduler":sched.state_dict(),"scaler":scaler.state_dict(),"classes":list(CLASSES),"stage":STAGE}
    torch.save(blob, os.path.join(CKPT_DIR, f"epoch_{epoch:03d}.pt"))
    torch.save(blob, os.path.join(CKPT_DIR, "last.pt"))
def _latest():
    c = []
    for base in [CKPT_DIR, RESUME_DIR]:
        if base and os.path.isdir(base):
            if os.path.exists(os.path.join(base,"last.pt")): c.append(os.path.join(base,"last.pt"))
            c += sorted(glob.glob(os.path.join(base,"epoch_*.pt")))
    return c[-1] if c else None
def maybe_resume(model, opt, sched, scaler):
    c = _latest()
    if not c: return 0, -1.0
    s = torch.load(c, map_location="cpu")
    model.load_state_dict(s["model"]); opt.load_state_dict(s["optimizer"])
    sched.load_state_dict(s["scheduler"]); scaler.load_state_dict(s["scaler"])
    print(f"[resume] {c} -> epoch {s['epoch']+1}")
    return s["epoch"]+1, s["best_f1"]


## 6. Train loop + evaluation

In [ ]:
from sklearn.metrics import f1_score, classification_report

LABELS = list(range(len(CLASSES)))  # pin both classes so metrics never crash on a one-class batch

@torch.inference_mode()
def evaluate(model, loader, device, crit):
    model.eval(); losses=[]; ys=[]; ps=[]
    for x,y in loader:
        x,y = x.to(device), y.to(device)
        lo = model(x); losses.append(crit(lo,y).item())
        ys.append(y.cpu().numpy()); ps.append(lo.argmax(1).cpu().numpy())
    yt,yp = np.concatenate(ys), np.concatenate(ps)
    return {"loss":float(np.mean(losses)),"acc":float((yt==yp).mean()),
            "macro_f1":float(f1_score(yt,yp,labels=LABELS,average="macro",zero_division=0)),
            "report":classification_report(yt,yp,labels=LABELS,target_names=CLASSES,output_dict=True,zero_division=0)}

def train(wandb=None):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(42); np.random.seed(42)
    loaders = {s: DataLoader(CoralPatchDataset(s), batch_size=BATCH_SIZE, shuffle=(s=="train"),
                 num_workers=2, pin_memory=True, drop_last=(s=="train")) for s in ("train","val","test")}
    model = DINOv2Classifier(len(CLASSES)).to(device)
    opt = torch.optim.AdamW(set_trainable(model, STAGE), weight_decay=0.05)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    scaler = torch.amp.GradScaler("cuda", enabled=(device=="cuda"))
    crit = nn.CrossEntropyLoss()
    start, best = maybe_resume(model, opt, sched, scaler)
    print(f"[train] stage={STAGE} device={device} trainable={sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
    for epoch in range(start, EPOCHS):
        model.train(); run=0.0
        for x,y in loaders["train"]:
            x,y = x.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=(device=="cuda")):
                loss = crit(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            run += loss.item()
        sched.step()
        tl = run/max(len(loaders["train"]),1)
        val = evaluate(model, loaders["val"], device, crit); best=max(best,val["macro_f1"])
        print(f"[epoch {epoch:03d}] train_loss={tl:.4f} val_acc={val['acc']:.4f} val_macroF1={val['macro_f1']:.4f} (best {best:.4f})")
        if wandb: wandb.log({"epoch":epoch,"train_loss":tl,"val_loss":val["loss"],
                             "val_acc":val["acc"],"val_macro_f1":val["macro_f1"],"lr":opt.param_groups[0]["lr"]})
        save_ckpt(model, opt, sched, scaler, epoch, best)
    res = {"best_val_macro_f1":best, "test":evaluate(model, loaders["test"], device, crit)}
    print(f"[test] acc={res['test']['acc']:.4f} macroF1={res['test']['macro_f1']:.4f}")
    json.dump({"stage":STAGE,"epochs":EPOCHS,"backbone":BACKBONE,"classes":list(CLASSES),
               "best_val_macro_f1":best,"test":{k:res['test'][k] for k in ('loss','acc','macro_f1')}},
              open(os.path.join(CKPT_DIR,"config.json"),"w"), indent=2)
    return model, res, device


## 7. ONE-SHOT: train → push weights → conformal → push

Single cell, so nothing is left to a forgotten step. Weights are pushed **immediately**
after training; then split-conformal (LAC, 90% coverage) is calibrated on the val split and
its realized coverage measured on test, and `conformal.json` is pushed too.

In [ ]:
# --- W&B ---
wb = None
if WANDB_KEY:
    import wandb; wandb.login(key=WANDB_KEY)
    wb = wandb; wandb.init(project=WANDB_PROJECT, name=f"dinov2b-{STAGE}",
                           config={"stage":STAGE,"epochs":EPOCHS,"batch_size":BATCH_SIZE})
model, results, device = train(wb)
if wb: wandb.finish()

from huggingface_hub import HfApi
def _push(local, remote):
    HfApi(token=os.environ.get("HF_TOKEN")).upload_file(
        path_or_fileobj=local, path_in_repo=remote, repo_id=HF_REPO)

# --- push weights immediately ---
if PUSH_TO_HUB and HF_REPO:
    from safetensors.torch import save_file
    HfApi(token=os.environ.get("HF_TOKEN")).create_repo(HF_REPO, exist_ok=True)
    w = os.path.join(CKPT_DIR, "model.safetensors"); save_file(model.state_dict(), w)
    _push(w, f"{STAGE}/model.safetensors")
    _push(os.path.join(CKPT_DIR, "config.json"), f"{STAGE}/config.json")
    print("✓ weights on the Hub")

# --- Phase 4: split-conformal (LAC, 90% coverage) ---
ALPHA = 0.10
@torch.inference_mode()
def _softmax(split):
    dl = DataLoader(CoralPatchDataset(split, train=False), batch_size=BATCH_SIZE, num_workers=2, pin_memory=True)
    model.eval(); P=[]; Y=[]
    for x,y in dl:
        P.append(torch.softmax(model(x.to(device)),1).cpu().numpy()); Y.append(y.numpy())
    return np.concatenate(P), np.concatenate(Y)

cal_P, cal_Y = _softmax("val"); test_P, test_Y = _softmax("test")
s = 1.0 - cal_P[np.arange(len(cal_Y)), cal_Y]; n = len(s)
qhat = float(np.quantile(s, min(np.ceil((n+1)*(1-ALPHA))/n, 1.0), method="higher"))
sets = test_P >= (1.0 - qhat); e = ~sets.any(1); sets[e, test_P[e].argmax(1)] = True
cov = float(sets[np.arange(len(test_Y)), test_Y].mean())
avg = float(sets.sum(1).mean()); unc = float((sets.sum(1)>1).mean())
print(f"\nqhat={qhat:.4f} n_cal={n} | coverage(test)={cov:.4f} (target {1-ALPHA:.0%}) | avg_set={avg:.3f} | uncertain={unc:.3f}")
calib = {"method":"LAC","alpha":ALPHA,"qhat":qhat,"n_cal":int(n),"classes":list(CLASSES),
         "empirical_coverage_test":cov,"avg_set_size_test":avg,
         "model_version":f"reefscan-dinov2-coral-v1-{STAGE.replace('_','')}"}
json.dump(calib, open(os.path.join(CKPT_DIR,"conformal.json"),"w"), indent=2)
if PUSH_TO_HUB and HF_REPO:
    _push(os.path.join(CKPT_DIR,"conformal.json"), f"{STAGE}/conformal.json")
    print("✓ conformal.json on the Hub")
print("\nDONE — weights + conformal saved%s." % (" + pushed to the Hub" if PUSH_TO_HUB else ""))
